# Calculation: re-planning heatmap, ghost landers, validation

Run the cells from top to bottom. Cells 4 and 5 only need to run **once** (results are saved in `data/derived/`, already in the repo). If Colab restarts, run cell 1 again.

## 1) Setup

In [ ]:
REPO_URL = "https://github.com/nkm1020/moonkeeper.git"
BRANCH = "main"
import os, sys
if not os.path.exists("/content/moonkeeper"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/moonkeeper
else:
    !git -C /content/moonkeeper pull -q
%pip install -q rasterio pyproj
os.chdir("/content/moonkeeper")
sys.path.insert(0, "pipeline")
print("working folder:", os.getcwd())
!ls data data/inputs data/outputs

In [ ]:
import json, csv, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
from moonkeeper_b import metrics, montecarlo, validation, geo, ephem, io_utils

## 2) Heatmap + Plan B (real inputs)
Thresholds come from `criteria.json` if it exists; otherwise the example values are used and the files are marked provisional.

In [ ]:
!python pipeline/run_heatmap.py data/inputs
!python pipeline/plot_heatmap.py data/outputs/heatmap.json
display(Image("data/outputs/heatmap.png"))

## 3) PGDA values at the 150 validation points (once)
NASA PGDA average Sun / Earth visibility maps, 85-90S, 60 m (Mazarico et al. 2011).

In [ ]:
import requests
base = "https://pgda.gsfc.nasa.gov/data/MoonIllumination/"
maps = {"sun": "AVGVISIB_85S_060M_201608.TIF", "earth": "AVGVISIB_85S_060M_201608_EARTH.TIF"}
raw = pathlib.Path("data/raw"); raw.mkdir(parents=True, exist_ok=True)
for name in maps.values():
    p = raw / name
    if not p.exists():
        p.write_bytes(requests.get(base + name, timeout=600).content)
        print("downloaded", name)
pts = list(csv.DictReader(open("data/points/validation_points_n150_seed42.csv")))
lat = np.array([float(r["lat_deg"]) for r in pts]); lon = np.array([float(r["lon_east_deg"]) for r in pts])
pgda_sun = validation.sample_geotiff(raw / maps["sun"], lat, lon)
pgda_earth = validation.sample_geotiff(raw / maps["earth"], lat, lon)
pathlib.Path("data/derived").mkdir(parents=True, exist_ok=True)
with open("data/derived/pgda_at_validation_points.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["point_id", "lat_deg", "lon_east_deg", "pgda_sun_frac", "pgda_earth_frac"])
    for r, s, e in zip(pts, pgda_sun, pgda_earth):
        w.writerow([r["point_id"], r["lat_deg"], r["lon_east_deg"], f"{s:.5f}", f"{e:.5f}"])
print(np.isfinite(pgda_sun).sum(), "points saved")

## 4) Long-term Sun and Earth directions, 18.6 years (once, about 10 Horizons calls)

In [ ]:
npz = pathlib.Path("data/derived/subpoints_2010_2028_6h.npz")
if not npz.exists():
    sp = ephem.fetch_subpoints("2010-01-01", "2028-08-08", step="6h")
    np.savez_compressed(npz, **{k: (v.astype(str) if k == "times" else v) for k, v in sp.items()})
print(npz, npz.stat().st_size // 1000, "kB")

## 5) Validation: our calculation vs NASA PGDA
Uses the ground-level (0 m) horizons for the 150 points, because the PGDA maps are for the surface.

In [ ]:
!python pipeline/run_validation_pgda.py --points data/points/validation_points_n150_seed42.csv \
    --horizons data/inputs/horizons_validation_n150_seed42.json \
    --pgda data/derived/pgda_at_validation_points.csv --subpoints data/derived/subpoints_2010_2028_6h.npz
summary = json.load(open("data/outputs/validation.json"))["data"]["pgda"]["summary"]
for k in ("sun", "earth"):
    s = summary[k]
    print(f"{k:5}  n={s['n']}  bias={s['bias']:+.3f}  RMSE={s['rmse']:.3f}  r={s['pearson_r']:.3f}")
display(Image("data/outputs/validation_pgda_sun.png")); display(Image("data/outputs/validation_pgda_earth.png"))

## 6) Ghost landers (im2, 317 grid points)
The landing date is the im2 date that passes with the **smallest power/comm margin above 0** (approach is one value for the whole grid, so it is left out of this choice). Set `LANDING` by hand to choose another date.

In [ ]:
grid_file = pathlib.Path("data/inputs/grid_horizons_im2.json")
assert grid_file.exists(), "grid_horizons_im2.json not in the repo yet (용휘's PR #1)"

hm = json.load(open("data/outputs/heatmap.json"))["data"]
i = hm["site_ids"].index("im2")
crit = {k: v for k, v in hm["criteria"].items() if v["metric"] != "approach"}
def margin(d):
    out = []
    for c in crit.values():
        v = hm["metrics"][c["metric"]][i][d]
        out.append((v - c["limit"]) / abs(c["limit"]) if c["kind"] == "min" else (c["limit"] - v) / abs(c["limit"]))
    return min(out)
cand = [(margin(d), t) for d, (t, p) in enumerate(zip(hm["landing_times_utc"], hm["passed"][i])) if p]
LANDING = min((m, t) for m, t in cand if m > 0)[1][:16]
print(len(cand), "passing dates for im2; using", LANDING)

!python pipeline/run_ghost_landers.py --site im2 --landing {LANDING} --grid data/points/grid_im2_r800_s80.csv --horizons {grid_file}
f = pathlib.Path("data/outputs") / f"ghost_landers_im2_{LANDING[:10].replace('-', '')}.json"
!cp {f} data/outputs/ghost_landers.json
g = json.load(open(f))["data"]
colors = {"ok": "tab:green", "power": "0.15", "approach": "tab:orange", "comm": "tab:blue", "other": "tab:red"}
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(g["grid"]["east_m"], g["grid"]["north_m"], c=["#d5ecd5" if p else "#ececec" for p in g["grid"]["passed"]], s=14, marker="s")
st = np.array(g["samples"]["status"])
for s, col in colors.items():
    k = st == s
    if k.any():
        ax.scatter(np.array(g["samples"]["east_m"])[k], np.array(g["samples"]["north_m"])[k], s=5, c=col, label=f"{s} ({k.mean():.0%})")
ax.set_aspect("equal"); ax.set_xlabel("East (m)"); ax.set_ylabel("North (m)"); ax.legend(markerscale=3, fontsize=8)
ax.set_title(f"Ghost landers, im2 {LANDING[:10]}: p={g['p']:.2f}, 95% CI [{g['ci95'][0]:.2f}, {g['ci95'][1]:.2f}]")
fig.savefig("data/outputs/ghost_landers_im2.png", dpi=120); plt.show()
print("grid points passing:", sum(g["grid"]["passed"]), "/", len(g["grid"]["passed"]), "| outside grid:", g["outside_grid_frac"])

## 7) Tests (optional)

In [ ]:
!python -m pytest -q tests